# R08-H51 - The parse is the bottleneck (multi-parser fidelity audit)

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R08 contrarian round, deterministic parser diff <br>
**Graph**: rebuilt CPAP graph (neo4j2, read-only) <br>

Registered test: a multi-parser fidelity audit (named-string preservation, numeric-token preservation)
over the ingested corpus. The project parser is `pymupdf4llm.to_markdown` (readers.read_document);
alternatives are pdfplumber and pypdf. Bar: >=20% of documents losing at least one entity-name string
(present in an alternative parser but lost by the project parser) CONFIRMS; named-string preservation
>=99% across parsers REFUTES (parsing vindicated). Also: does a persistent failure (unlinked mode-family
/ OSA) root upstream in parse loss?

In [1]:
import json, datetime, hashlib, re, collections
from pathlib import Path
from neo4j import GraphDatabase
from rich import print as rprint
NEO4J_URI='bolt://user-konrad.jelen-kgf-neo4j2:7687'
PDFDIR=Path('../data/external/cpap-datasheets-and-manuals')
def docid(name): return 'd_'+hashlib.sha1(name.encode()).hexdigest()[:16]
def norm(x): return ' '.join((x or '').lower().split())
def nospace(x): return re.sub(r'\s+','',(x or '').lower())

## Extract each PDF with three parsers

In [2]:
import pymupdf4llm, pdfplumber
from pypdf import PdfReader
import warnings; warnings.filterwarnings('ignore')

drv=GraphDatabase.driver(NEO4J_URI, auth=('neo4j','kgfoundry'), notifications_min_severity='OFF')
with drv.session() as s:
    docnames=[r['n'] for r in s.run('MATCH (k:KGFDocument) RETURN k.name AS n')]
rprint(f'{len(docnames)} ingested documents')

texts={}  # docname -> {parser: text}
for nm in docnames:
    p=PDFDIR/nm
    d={}
    try: d['pymupdf4llm']=pymupdf4llm.to_markdown(str(p))
    except Exception as e: d['pymupdf4llm']=''; rprint('pymupdf4llm FAIL',nm,e)
    try:
        with pdfplumber.open(str(p)) as pdf:
            d['pdfplumber']='\n'.join((pg.extract_text() or '') for pg in pdf.pages)
    except Exception as e: d['pdfplumber']=''; rprint('pdfplumber FAIL',nm,e)
    try:
        d['pypdf']='\n'.join((pg.extract_text() or '') for pg in PdfReader(str(p)).pages)
    except Exception as e: d['pypdf']=''; rprint('pypdf FAIL',nm,e)
    texts[nm]=d
rprint('extracted. char counts (project/plumber/pypdf) sample:')
for nm in docnames[:5]:
    rprint(f'  {nm[:40]:40s}', {k:len(v) for k,v in texts[nm].items()})

27 ingested documents

Ignoring wrong pointing object 6 0 (offset 0)


Ignoring wrong pointing object 8 0 (offset 0)


Ignoring wrong pointing object 10 0 (offset 0)


Ignoring wrong pointing object 12 0 (offset 0)


...

0-20190113114505.pdf                    
{'pymupdf4llm': 4881, 'pdfplumber': 4340, 'pypdf': 4370}

1017900r4_ResMed_Product_Catalogue_ANZ_E
{'pymupdf4llm': 14762, 'pdfplumber': 13448, 'pypdf': 13848}

3B_User-Manual_CPAP-Auto-CPAP_RESmart_BM
{'pymupdf4llm': 52638, 'pdfplumber': 46729, 'pypdf': 49683}

ARTP_Standards_of_Care_-_CPAP_Devices_(T
{'pymupdf4llm': 85639, 'pdfplumber': 82090, 'pypdf': 82866}

Airsense-Brochure.pdf                   
{'pymupdf4llm': 6132, 'pdfplumber': 8952, 'pypdf': 9084}

## Named-string preservation per parser

In [3]:
# normalized text per doc per parser
ntext={nm:{p:norm(t) for p,t in d.items()} for nm,d in texts.items()}
nstext={nm:{p:nospace(t) for p,t in d.items()} for nm,d in texts.items()}
id2name={docid(nm):nm for nm in docnames}

with drv.session() as s:
    ents=s.run('MATCH (e:Entity) WHERE e.source_documents IS NOT NULL '
               'RETURN e.id AS id, e.name AS name, labels(e) AS types, e.source_documents AS sd').data()
drv.close()

PARSERS=['pymupdf4llm','pdfplumber','pypdf']
def present(name, docnm, parser):
    nn=norm(name)
    if nn and nn in ntext[docnm][parser]: return True
    # letter-spacing / stylized fallback: whitespace-stripped
    ns=nospace(name)
    return bool(ns) and ns in nstext[docnm][parser]

# per entity: found in ANY of its source docs, per parser
rows=[]
for e in ents:
    docs=[id2name[d] for d in e['sd'] if d in id2name]
    if not docs: continue
    found={p: any(present(e['name'],dn,p) for dn in docs) for p in PARSERS}
    rows.append(dict(id=e['id'],name=e['name'],types=e['types'],docs=docs,found=found))
rprint(f'entities with resolvable source docs: {len(rows)}')
for p in PARSERS:
    pres=sum(r['found'][p] for r in rows)/len(rows)
    rprint(f'  {p:14s} named-string preservation: {pres:.1%}')
union=sum(any(r['found'].values()) for r in rows)/len(rows)
rprint(f'[bold]union (present in >=1 parser): {union:.1%}[/bold]')
rprint(f'absent from ALL parsers (extraction synthesis, not parse loss): {sum(not any(r["found"].values()) for r in rows)}')

entities with resolvable source docs: 2798

pymupdf4llm    named-string preservation: 72.4%

pdfplumber     named-string preservation: 72.1%

pypdf          named-string preservation: 75.1%

union (present in >=1 parser): 75.8%

absent from ALL parsers (extraction synthesis, not parse loss): 676

## Parse loss - names an alternative preserves but the project parser drops

In [4]:
# true parse loss: name present in some alternative parser but NOT in pymupdf4llm
parse_lost=[r for r in rows if not r['found']['pymupdf4llm'] and (r['found']['pdfplumber'] or r['found']['pypdf'])]
rprint(f'[bold]entities parse-lost by project parser (recovered by an alternative): {len(parse_lost)}[/bold]')
# per-document loss
docs_losing=collections.Counter()
for r in parse_lost:
    for dn in r['docs']:
        # only credit the doc where the alternative found it and project did not
        if not present(r['name'],dn,'pymupdf4llm') and (present(r['name'],dn,'pdfplumber') or present(r['name'],dn,'pypdf')):
            docs_losing[dn]+=1
frac_docs_losing=len(docs_losing)/len(docnames)
rprint(f'[bold]documents losing >=1 entity-name string to parsing: {len(docs_losing)}/{len(docnames)} = {frac_docs_losing:.1%}[/bold]')
for dn,c in docs_losing.most_common():
    rprint(f'  {c:3d} lost  {dn}')
rprint('sample parse-lost names:', [r['name'] for r in parse_lost[:20]])

entities parse-lost by project parser (recovered by an alternative): 95

documents losing >=1 entity-name string to parsing: 18/27 = 66.7%

22 lost  Resvent-iBreeze-Auto-CPAP-User-Manual.pdf

21 lost  moh-adp-product-manual-respiratory-devices-airway-clearance-en-2023-06-14.pdf

19 lost  Sleep And Respiratory Medical Devices Brochure.pdf

...

1 lost  Clinical-Job-Aid_bCPAP-Diamedica_Final_07-05-2024.pdf

1 lost  DSDC-CPAP-Therapy-Catalogue.pdf

1 lost  DreamStation_CPAP_User_Manual.pdf

1 lost  ResMed-Airsense-11-Manual.pdf

sample parse-lost names:
[
    'Central sleep apnea detection',
    'Anti-asphyxia Valve',
    'Automatic Positive Airway Pressure',
    'AutoSet algorithm',
    'Enhanced Climate Control',
    'DreamStation humidifier',
    'RESmart Auto CPAP System',
    'European CE Declaration of Conformity',
    'RESmart Auto',
    'Multilevel Filtration-Purification System',
    'Oxygen Gas Flow Column',
    'Diamedica Bubble CPAP',
    'improved forehead support',
    'East Meets West Breath of Life program',
    'Leak compensation technology',
    'PATH Bubble CPAP Kit',
    'Bluetooth Wireless Technology',
    'Communications Connector',
    'Medical Product Note',
    'Ultra-fine filter'
]

## Numeric-token preservation (33 gold evidence strings vs their source PDFs)

In [5]:
import yaml
probes=yaml.safe_load(open('../tests/probes/cpap-probe-set.yml'))
gold_checks=[]
for pr in probes:
    for g in (pr.get('gold_evidence') or []):
        for src in (pr.get('sources') or []):
            if src in texts:
                gold_checks.append((g,src))
rprint(f'{len(gold_checks)} (gold-string, source-doc) checks')
for p in PARSERS:
    ok=sum(1 for g,src in gold_checks if norm(g) in ntext[src][p] or nospace(g) in nstext[src][p])
    rprint(f'  {p:14s} gold-string preservation: {ok}/{len(gold_checks)} = {ok/len(gold_checks):.1%}')
gold_lost_project=[(g,src) for g,src in gold_checks
    if not (norm(g) in ntext[src]['pymupdf4llm'] or nospace(g) in nstext[src]['pymupdf4llm'])
    and any(norm(g) in ntext[src][p] or nospace(g) in nstext[src][p] for p in ['pdfplumber','pypdf'])]
rprint(f'gold strings lost by project parser but recovered by an alternative: {gold_lost_project}')

49 (gold-string, source-doc) checks

pymupdf4llm    gold-string preservation: 33/49 = 67.3%

pdfplumber     gold-string preservation: 33/49 = 67.3%

pypdf          gold-string preservation: 33/49 = 67.3%

gold strings lost by project parser but recovered by an alternative: []

## Root-cause probe - mode-family (SleepStyle) and OSA classes

In [6]:
targets=[r for r in rows if re.search(r'sleepstyle|mode|osa|obstructive|apnea', r['name'], re.I)]
mode_family=[r for r in rows if re.search(r'sleepstyle', r['name'], re.I)]
rprint(f'SleepStyle-named entities: {len(mode_family)}')
for r in mode_family:
    rprint(f"  {r['name']!r} found={r['found']} docs={[d[:30] for d in r['docs']]}")
# is the document subject string present at all in its own text
ss_doc='SleepStyle_200_Operating_Manual.pdf'
if ss_doc in texts:
    for probe in ['SleepStyle 200','SleepStyle','200 Series']:
        rprint(f'  {probe!r} in SleepStyle manual:', {p:(norm(probe) in ntext[ss_doc][p] or nospace(probe) in nstext[ss_doc][p]) for p in PARSERS})
osa=[r for r in rows if re.search(r'\bosa\b|obstructive sleep', r['name'], re.I)]
rprint(f'OSA-named entities: {len(osa)}:', [r['name'] for r in osa])

SleepStyle-named entities: 10

'Fisher & Paykel SleepStyle' found={'pymupdf4llm': True, 'pdfplumber': True, 'pypdf': True} docs=['Sleep And 
Respiratory Medical ']

'SleepStyle Auto' found={'pymupdf4llm': False, 'pdfplumber': False, 'pypdf': False} docs=['Sleep And Respiratory 
Medical ']

'SleepStyle CPAP' found={'pymupdf4llm': False, 'pdfplumber': False, 'pypdf': False} docs=['Sleep And Respiratory 
Medical ']

'SleepStyle extended care CPAP warranty card' found={'pymupdf4llm': False, 'pdfplumber': False, 'pypdf': False} 
docs=['Sleep And Respiratory Medical ']

...

'F&P Sleepstyle Auto CPAP - no modem model - SPSABN' found={'pymupdf4llm': False, 'pdfplumber': False, 'pypdf': 
True} docs=['moh-adp-product-manual-respira']

'SleepStyle 200' in SleepStyle manual:
{'pymupdf4llm': False, 'pdfplumber': False, 'pypdf': False}

'SleepStyle' in SleepStyle manual:
{'pymupdf4llm': False, 'pdfplumber': True, 'pypdf': True}

'200 Series' in SleepStyle manual:
{'pymupdf4llm': True, 'pdfplumber': True, 'pypdf': True}

OSA-named entities: 4:
[
    'Obstructive Sleep Apnea',
    'Obstructive Sleep Apnoea/Hypopnoea Syndrome',
    'mild to moderate OSA',
    'Fundamentals of obstructive sleep apnea'
]

## Verdict + report

In [7]:
confirm = frac_docs_losing>=0.20
refute = union>=0.99 and len(parse_lost)==0
if confirm: verdict='CONFIRMED'; reason=f'{frac_docs_losing:.1%} of documents lose >=1 entity-name string to parsing (>=20% bar)'
elif refute: verdict='REFUTED'; reason=f'named-string union preservation {union:.1%} (>=99%) and zero parse-loss - parsing vindicated'
else: verdict='REFUTED (weak)'; reason=f'only {frac_docs_losing:.1%} of documents show parse loss (<20% bar); union preservation {union:.1%}'
report=dict(hypothesis='R08-H51', parsers=PARSERS, n_documents=len(docnames), n_entities_checked=len(rows),
  named_string_preservation={p:sum(r['found'][p] for r in rows)/len(rows) for p in PARSERS},
  union_preservation=union,
  absent_from_all_parsers=sum(not any(r['found'].values()) for r in rows),
  entities_parse_lost_by_project=len(parse_lost),
  parse_lost_names=[r['name'] for r in parse_lost],
  documents_losing_entity_name=len(docs_losing), frac_documents_losing=frac_docs_losing,
  docs_losing_detail=dict(docs_losing),
  gold_string_preservation={p:sum(1 for g,src in gold_checks if norm(g) in ntext[src][p] or nospace(g) in nstext[src][p])/len(gold_checks) for p in PARSERS},
  gold_lost_by_project=gold_lost_project,
  sleepstyle_entities=[dict(name=r['name'],found=r['found']) for r in mode_family],
  bar='>=20% docs lose >=1 entity name confirms; >=99% union preservation refutes',
  verdict=verdict, reason=reason)
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
path=f'../reports/parse-fidelity-h51-{stamp}.json'
json.dump(report,open(path,'w'),indent=2)
rprint(f'[bold green]{verdict}[/bold green] - {reason}')
rprint('wrote',path)

CONFIRMED - 66.7% of documents lose >=1 entity-name string to parsing (>=20% bar)

wrote ../reports/parse-fidelity-h51-20260707-093507.json